# **MODELS**

In [1]:
from dataclasses import dataclass
"""
Represents a structured product documnet used in a RAG pipeline.
This documnet serves as the souce of truth for product information.
It stores product information before it is converted into text for chunking and embedding.
"""
@dataclass
class Document:
    document_id : int
    name : str
    category : str
    features : list[str]
    specifications: dict[str,str]
    description: str

    def to_text(self)->str:
        features_text = "\n".join(f"- {feature}" for feature in self.features)
        specifications_text = "\n".join(f"{key}: {value}" for key,value in self.specifications.items())
        return f"""Product Name: {self.name}
        Category: {self.category}
        Features: {features_text}
         Specifications: {specifications_text}
        Description: {self.description}"""
    def get_sections(self)->dict[str,str]:
        """
        Returns the doucumnet organized into semantic sections.
        This representaion is used for field-aware chunking.
        """
        features_text = "\n".join(f"- {feature}" for feature in self.features)
        specifications_text = "\n".join(f"{key}:{value}" for key,value in self.specifications.items())

        return {
            "identity":(
                f"Product Name:{self.name}\n",
                f"Category: {self.category}"
            ),
            "features": features_text,
            "specifications": specifications_text,
            "description": self.description
        }        

In [2]:
@dataclass
class Chunk:
    """
    Represnts a retreivable chunk generated form a product document.
    """
    chunk_id: int
    document_id: int
    text: str
    metadata: dict[str,str]
    chunk_index: int

## **Base Chunker**

In [3]:
from abc import ABC,abstractmethod

class BaseChunker(ABC):
    """
    Abstract Base Class for all chunking strategies.

    Every upcoming chunker implementaion must convert a
    Document into a list of chunk objects.

    """
    @abstractmethod
    def chunk(self,document: Document) -> list[Chunk]:
        """
        Split the doc. into retrievable chunks.
        Args: document - represnts the doc. to be chunked.
        Returns: A list of chunk objects.
        """
        pass

class FixedSizeChunker(BaseChunker):
    """
    splits a doc. with fixed size chunks with overlap.
    """
    def __init__(self,chunk_size:int, chunk_overlap: int = 0):
        if chunk_size <=0:
            raise ValueError("chunk_size must be greater than 0.")
        if chunk_overlap < 0:
            raise ValueError("chunk_overlap cannot be negative.")
        if chunk_overlap >=chunk_size:
            raise ValueError("chunk_overlap must be smaller than chunk_size.")
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

    def chunk(self,document:Document) -> list[Chunk]:
        text = document.to_text()
        chunks = []
        start = 0
        chunk_index = 0
        while start < len(text):
            end = min(start + self.chunk_size,len(text))
            chunk_text = text[start:end]
            chunk = Chunk(
                chunk_id = f"{document.document_id}_{chunk_index}",
                document_id = document.document_id,
                text = chunk_text,
                metadata = {"category": document.category},
                chunk_index = chunk_index
            )
            chunks.append(chunk)

            if end == len(text):
                break
            start = end -self.chunk_overlap
            chunk_index += 1
        return chunks

class FieldAwareChunker(BaseChunker):
    """
    Splits a structured product documnet into sematically meaningful chunks
    based on its fields.
    """
    def chunk(self,document: Document) ->list[Chunk]:
        sections = document.get_sections()
        chunks = []

        for index,(section_name,section_text) in enumerate(sections.items()):
            chunk = Chunk(
                chunk_id = f"{document.document_id}_{index}",
                document_id = document.document_id,
                text = section_text,
                metadata={
                    "category": document.category,
                    "section": section_name
                },
                chunk_index=index
            )
            chunks.append(chunk)
        return chunks


In [4]:
#TESTING

In [5]:
sample_doc = Document(
    document_id=1,
    name="Titan Gaming Laptop",
    category="Laptop",
    features=[
        "RTX 4080 GPU",
        "32GB DDR5 RAM",
        "1TB NVMe SSD",
        "WiFi 7"
    ],
    specifications={
        "Processor": "Intel Core Ultra 9",
        "Display": "16-inch QHD",
        "Battery": "90Wh",
        "Weight": "2.4 kg"
    },
    description=(
        "The Titan Gaming Laptop is designed for gamers and creators. "
        "It delivers excellent gaming performance, fast rendering speeds, "
        "and long battery life while maintaining an efficient cooling system."
    )
)


In [6]:
fixed_chunker = FixedSizeChunker(
    chunk_size=120,
    chunk_overlap=20
)
fixed_chunks = fixed_chunker.chunk(sample_doc)

In [7]:
print(f"Total Chunks: {len(fixed_chunks)}\n")

for chunk in fixed_chunks:
    print("=" * 60)
    print(f"Chunk ID      : {chunk.chunk_id}")
    print(f"Document ID   : {chunk.document_id}")
    print(f"Chunk Index   : {chunk.chunk_index}")
    print(f"Metadata      : {chunk.metadata}")
    print("\nChunk Text:")
    print(chunk.text)

Total Chunks: 5

Chunk ID      : 1_0
Document ID   : 1
Chunk Index   : 0
Metadata      : {'category': 'Laptop'}

Chunk Text:
Product Name: Titan Gaming Laptop
        Category: Laptop
        Features: - RTX 4080 GPU
- 32GB DDR5 RAM
- 1TB NVMe S
Chunk ID      : 1_1
Document ID   : 1
Chunk Index   : 1
Metadata      : {'category': 'Laptop'}

Chunk Text:
DR5 RAM
- 1TB NVMe SSD
- WiFi 7
         Specifications: Processor: Intel Core Ultra 9
Display: 16-inch QHD
Battery: 90W
Chunk ID      : 1_2
Document ID   : 1
Chunk Index   : 2
Metadata      : {'category': 'Laptop'}

Chunk Text:
nch QHD
Battery: 90Wh
Weight: 2.4 kg
        Description: The Titan Gaming Laptop is designed for gamers and creators. I
Chunk ID      : 1_3
Document ID   : 1
Chunk Index   : 3
Metadata      : {'category': 'Laptop'}

Chunk Text:
mers and creators. It delivers excellent gaming performance, fast rendering speeds, and long battery life while maintain
Chunk ID      : 1_4
Document ID   : 1
Chunk Index   : 4
Metadata   

In [8]:
field_chunker = FieldAwareChunker()

field_chunks = field_chunker.chunk(sample_doc)

In [9]:
print(f"Total Chunks: {len(field_chunks)}\n")

for chunk in field_chunks:
    print("=" * 60)
    print(f"Chunk ID      : {chunk.chunk_id}")
    print(f"Chunk Index   : {chunk.chunk_index}")
    print(f"Metadata      : {chunk.metadata}")
    print("\nChunk Text:")
    print(chunk.text)

Total Chunks: 4

Chunk ID      : 1_0
Chunk Index   : 0
Metadata      : {'category': 'Laptop', 'section': 'identity'}

Chunk Text:
('Product Name:Titan Gaming Laptop\n', 'Category: Laptop')
Chunk ID      : 1_1
Chunk Index   : 1
Metadata      : {'category': 'Laptop', 'section': 'features'}

Chunk Text:
- RTX 4080 GPU
- 32GB DDR5 RAM
- 1TB NVMe SSD
- WiFi 7
Chunk ID      : 1_2
Chunk Index   : 2
Metadata      : {'category': 'Laptop', 'section': 'specifications'}

Chunk Text:
Processor:Intel Core Ultra 9
Display:16-inch QHD
Battery:90Wh
Weight:2.4 kg
Chunk ID      : 1_3
Chunk Index   : 3
Metadata      : {'category': 'Laptop', 'section': 'description'}

Chunk Text:
The Titan Gaming Laptop is designed for gamers and creators. It delivers excellent gaming performance, fast rendering speeds, and long battery life while maintaining an efficient cooling system.


## Synthetic Dataset Generation

In [20]:
import random
class SyntheticDatasetGenerator:
    """
    Generate a synthetic corpus of realistic product documents for a RAG pipeline.
    The generator has differnt profiles, diverse and no duplicates.
    """
    def __init__(self)->None:
        self.catalog = {
            "Laptop": {
                "Gaming": {
                    "brands": ["ASUS", "MSI", "Lenovo", "Alienware"],
                    "features": [
                        "AI-enhanced performance",
                        "Advanced thermal cooling",
                        "High refresh-rate display",
                        "Premium build quality",
                        "Designed for AAA gaming",
                        "Fast SSD storage"
                    ],
                    "attributes": {
                        "Processor": ["Intel Core Ultra 9", "AMD Ryzen 9"],
                        "Graphics": ["RTX 4070", "RTX 4080"],
                        "Memory": ["32GB", "64GB"],
                        "Storage": ["1TB SSD", "2TB SSD"],
                        "Battery": ["80Wh", "90Wh"]
                    }
                },
                "Business": {
                    "brands": ["Dell", "HP", "Lenovo"],
                    "features": [
                        "Lightweight design",
                        "Long battery life",
                        "Enterprise-grade security",
                        "Fast multitasking",
                        "Ideal for professionals",
                        "Reliable everyday performance"
                    ],
                    "attributes": {
                        "Processor": ["Intel Core Ultra 7", "AMD Ryzen 7"],
                        "Graphics": ["Intel Arc", "Intel Iris Xe"],
                        "Memory": ["16GB", "32GB"],
                        "Storage": ["512GB SSD", "1TB SSD"],
                        "Battery": ["70Wh", "80Wh"]
                    }
                },
                "Budget": {
                    "brands": ["Acer", "ASUS", "HP"],
                    "features": [
                        "Affordable pricing",
                        "Energy efficient",
                        "Compact design",
                        "Reliable everyday computing",
                        "Student friendly"
                    ],
                    "attributes": {
                        "Processor": ["Intel Core i3", "AMD Ryzen 5"],
                        "Graphics": ["Intel UHD"],
                        "Memory": ["8GB", "16GB"],
                        "Storage": ["256GB SSD", "512GB SSD"],
                        "Battery": ["50Wh", "60Wh"]
                    }
                }
            },

            "Smartphone": {
                "Flagship": {
                    "brands": ["Samsung", "Apple", "Google"],
                    "features": [
                        "Professional-grade camera system",
                        "Ultra-smooth AMOLED display",
                        "Fast wireless charging",
                        "5G connectivity",
                        "AI-powered photography",
                        "Premium flagship experience"
                    ],
                    "attributes": {
                        "Chipset": ["Snapdragon 8 Elite", "Apple A18 Pro", "Tensor G5"],
                        "Display": ["6.7-inch AMOLED"],
                        "Storage": ["256GB", "512GB"],
                        "Battery": ["5000mAh"],
                        "Camera": ["50MP Triple Camera"]
                    }
                },
                "Budget": {
                    "brands": ["Redmi", "POCO", "Realme"],
                    "features": [
                        "Excellent value for money",
                        "Long battery life",
                        "Smooth everyday performance",
                        "Modern design",
                        "Fast charging support"
                    ],
                    "attributes": {
                        "Chipset": ["Snapdragon 6 Gen 1", "Helio G99"],
                        "Display": ["6.5-inch LCD"],
                        "Storage": ["128GB"],
                        "Battery": ["5000mAh"],
                        "Camera": ["50MP Dual Camera"]
                    }
                }
            },

            "Tablet": {
                "Premium": {
                    "brands": ["Apple", "Samsung"],
                    "features": [
                        "Large immersive display",
                        "Perfect for creativity",
                        "Excellent multimedia experience",
                        "Powerful multitasking"
                    ],
                    "attributes": {
                        "Chipset": ["Apple M4", "Snapdragon X Elite"],
                        "Display": ["12.9-inch OLED"],
                        "Storage": ["256GB", "512GB"],
                        "Battery": ["9000mAh"]
                    }
                }
            },

            "Monitor": {
                "Gaming": {
                    "brands": ["LG", "ASUS", "MSI"],
                    "features": [
                        "Ultra-smooth gameplay",
                        "High refresh-rate display",
                        "Low response time",
                        "Immersive viewing experience"
                    ],
                    "attributes": {
                        "Resolution": ["1440p", "4K"],
                        "Refresh Rate": ["165Hz", "240Hz"],
                        "Panel": ["IPS", "OLED"],
                        "Size": ["27-inch", "32-inch"]
                    }
                }
            },

            "Keyboard": {
                "Mechanical": {
                    "brands": ["Keychron", "Corsair", "Logitech"],
                    "features": [
                        "Tactile typing experience",
                        "Customizable RGB lighting",
                        "Durable mechanical switches",
                        "Comfortable for long sessions"
                    ],
                    "attributes": {
                        "Switch Type": ["Red", "Brown", "Blue"],
                        "Layout": ["TKL", "Full Size", "75%"],
                        "Connectivity": ["USB-C", "Bluetooth"],
                        "Backlight": ["RGB", "White"]
                    }
                }
            },

            "Mouse": {
                "Wireless": {
                    "brands": ["Logitech", "Razer", "SteelSeries"],
                    "features": [
                        "Precision tracking",
                        "Ergonomic design",
                        "Low latency wireless connection",
                        "Long battery life"
                    ],
                    "attributes": {
                        "Sensor": ["Optical", "Laser"],
                        "DPI": ["16000", "26000"],
                        "Connectivity": ["2.4GHz", "Bluetooth"]
                    }
                }
            },

            "Headphones": {
                "Wireless": {
                    "brands": ["Sony", "Bose", "Sennheiser"],
                    "features": [
                        "Immersive audio",
                        "Active noise cancellation",
                        "Comfortable all-day wear",
                        "Crystal-clear voice calls"
                    ],
                    "attributes": {
                        "Driver Size": ["40mm", "45mm"],
                        "Noise Cancellation": ["Yes"],
                        "Battery Life": ["30 Hours", "40 Hours"]
                    }
                }
            },

            "Smartwatch": {
                "Fitness": {
                    "brands": ["Apple", "Samsung", "Garmin"],
                    "features": [
                        "Comprehensive health tracking",
                        "Built-in GPS",
                        "Water resistant",
                        "Long-lasting battery"
                    ],
                    "attributes": {
                        "Display": ["AMOLED"],
                        "Battery Life": ["2 Days", "5 Days"],
                        "Water Resistance": ["5 ATM"],
                        "Sensors": ["Heart Rate", "SpO2", "GPS"]
                    }
                }
            }
        }

        self.used_signatures: set[tuple] = set()
        self.current_id: int = 1

    

    def generate_document(self) -> Document:
        """
        Generates a single realistic product document.
        """

       # Select category and profile
        category = random.choice(list(self.catalog.keys()))
        profile = random.choice(list(self.catalog[category].keys()))

        pool = self.catalog[category][profile]

        brand = random.choice(pool["brands"])

        # Generate specifications dynamically
        specifications = {}

        for attribute, values in pool["attributes"].items():
            specifications[attribute] = random.choice(values)

        # Duplicate signature
        signature = (
            category,
            profile,
            brand,
            tuple(sorted(specifications.items()))
        )

        if signature in self.used_signatures:
            return self.generate_document()

        self.used_signatures.add(signature)

        # Product name
        name = f"{brand} {profile} {category}"

        # Features 
        num_features = min(4, len(pool["features"]))

        features = random.sample(
            pool["features"],
            k=num_features
        )

        # Generic description
        description = (
            f"The {name} is a {profile.lower()} {category.lower()} "
            f"designed for users seeking {', '.join(features[:-1])}, "
            f"and {features[-1]}. "
            f"It delivers reliable performance and a premium user experience."
        )
        document = Document(
            document_id=self.current_id,
            name=name,
            category=category,
            features=features,
            specifications=specifications,
            description=description
        )

        self.current_id += 1

        return document
    def generate_dataset(self,num_documents: int) ->list[Document]:
        """
        Generate a synthetic dataset containing the specified number of 
        product documents.
        """
        dataset = []
        for i in range(num_documents):
            dataset.append(self.generate_document())

        return dataset

In [21]:
generator = SyntheticDatasetGenerator()

dataset = generator.generate_dataset(3)

for doc in dataset:
    print(doc)
    print("-" * 80)

Document(document_id=1, name='Apple Flagship Smartphone', category='Smartphone', features=['Professional-grade camera system', 'Ultra-smooth AMOLED display', 'Fast wireless charging', 'AI-powered photography'], specifications={'Chipset': 'Tensor G5', 'Display': '6.7-inch AMOLED', 'Storage': '256GB', 'Battery': '5000mAh', 'Camera': '50MP Triple Camera'}, description='The Apple Flagship Smartphone is a flagship smartphone designed for users seeking Professional-grade camera system, Ultra-smooth AMOLED display, Fast wireless charging, and AI-powered photography. It delivers reliable performance and a premium user experience.')
--------------------------------------------------------------------------------
Document(document_id=2, name='Alienware Gaming Laptop', category='Laptop', features=['Fast SSD storage', 'AI-enhanced performance', 'High refresh-rate display', 'Designed for AAA gaming'], specifications={'Processor': 'AMD Ryzen 9', 'Graphics': 'RTX 4080', 'Memory': '64GB', 'Storage': '